# Avaliação do consumo de memória — Dashboard ENEM

**Escopo:** diagnóstico dos pontos 1 e 2 da revisão de memória.

1. **Carregamento de dados:** tamanho dos Parquets, esquema, leitura integral versus leitura de colunas selecionadas e possibilidade de filtrar antes de materializar os DataFrames.
2. **Processamento:** custo de cópias, objetos intermediários e manutenção simultânea das tabelas em memória.

O notebook é diagnóstico: **não altera `data_loader.py`, os Parquets nem o dashboard**. Execute as células em ordem, a partir da pasta `app_render`.

> A primeira etapa identifica os caminhos e as colunas disponíveis. A seleção de colunas para teste deve ser definida depois de conferir quais campos são realmente utilizados pelas páginas do dashboard.

## 1. Preparação

In [20]:
from pathlib import Path
import os
import gc
import time
import json
import platform
import sys

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

try:
    import psutil
except ImportError:
    psutil = None

RAIZ = Path.cwd()
if RAIZ.name != "app_render" and (RAIZ / "app_render").is_dir():
    RAIZ = RAIZ / "app_render"

DIRETORIO_DADOS = RAIZ / "data" / "consumo" / "uf=PA"

ARQUIVOS = {
    "painel": DIRETORIO_DADOS / "painel_municipal.parquet",
    "faixa": DIRETORIO_DADOS / "faixa_etaria_municipal.parquet",
    "grupo": DIRETORIO_DADOS / "desempenho_grupos_municipal.parquet",
    "dsp": DIRETORIO_DADOS / "desempenho_sexo_faixa_etaria_municipal.parquet",
}

print("Raiz:", RAIZ.resolve())
print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("PyArrow:", pa.__version__)
print("Sistema:", platform.platform())
print("\nArquivos:")
for nome, caminho in ARQUIVOS.items():
    print(f"{nome:10} | existe={caminho.exists()} | {caminho}")

Raiz: /home/akel/PycharmProjects/Projeto_ENEM/app_render
Python: 3.11.18
Pandas: 3.1.0
PyArrow: 26.0.1
Sistema: Linux-6.8.0-149-generic-x86_64-with-glibc2.39

Arquivos:
painel     | existe=True | /home/akel/PycharmProjects/Projeto_ENEM/app_render/data/consumo/uf=PA/painel_municipal.parquet
faixa      | existe=True | /home/akel/PycharmProjects/Projeto_ENEM/app_render/data/consumo/uf=PA/faixa_etaria_municipal.parquet
grupo      | existe=True | /home/akel/PycharmProjects/Projeto_ENEM/app_render/data/consumo/uf=PA/desempenho_grupos_municipal.parquet
dsp        | existe=True | /home/akel/PycharmProjects/Projeto_ENEM/app_render/data/consumo/uf=PA/desempenho_sexo_faixa_etaria_municipal.parquet


## 2. Instrumentação de memória

In [2]:
def memoria_processo_mb():
    if psutil is None:
        return np.nan
    processo = psutil.Process(os.getpid())
    return processo.memory_info().rss / (1024 ** 2)

def tamanho_df_mb(df):
    return df.memory_usage(index=True, deep=True).sum() / (1024 ** 2)

def registrar_memoria(etapa):
    valor = memoria_processo_mb()
    print(f"{etapa:<45} RSS={valor:,.1f} MB" if pd.notna(valor)
          else f"{etapa:<45} RSS indisponível (instale psutil)")
    return valor

rss_inicial = registrar_memoria("Kernel inicial")

Kernel inicial                                RSS=138.8 MB


## 3. Inventário dos Parquets — tamanho e esquema

In [3]:
inventario = []
esquemas = {}

for nome, caminho in ARQUIVOS.items():
    if not caminho.exists():
        print(f"Arquivo ausente: {caminho}")
        continue

    parquet = pq.ParquetFile(caminho)
    schema = parquet.schema_arrow
    esquemas[nome] = schema

    inventario.append({
        "tabela": nome,
        "arquivo_MB": caminho.stat().st_size / (1024 ** 2),
        "linhas": parquet.metadata.num_rows,
        "colunas": len(schema.names),
        "row_groups": parquet.metadata.num_row_groups,
    })

    print(f"\n### {nome}")
    print(f"Linhas: {parquet.metadata.num_rows:,} | Colunas: {len(schema.names)}")
    print(schema)

inventario_df = pd.DataFrame(inventario)
display(inventario_df)


### painel
Linhas: 6,464 | Colunas: 82
ano: int64
CO_MUNICIPIO_TERRITORIO: large_string
NO_MUNICIPIO_TERRITORIO: large_string
TIPO_REFERENCIA_TERRITORIAL: large_string
n_participantes: int64
n_masculino: int64
n_feminino: int64
n_faixa_etaria_informada: int64
pct_masculino: double
pct_feminino: double
n_registros_presenca: int64
n_presentes: int64
n_ausentes: int64
n_presenca_informada: int64
pct_presenca: double
pct_ausencia: double
n_validos_redacao: int64
n_zeros_redacao: int64
media_redacao: double
media_positivas_redacao: double
mediana_redacao: double
std_redacao: double
q1_redacao: double
q3_redacao: double
n_avaliados_redacao: int64
n_validos_objetiva: double
n_zeros_objetiva: double
media_objetiva: double
media_positivas_objetiva: double
mediana_objetiva: double
std_objetiva: double
q1_objetiva: double
q3_objetiva: double
n_avaliados_objetiva: double
n_validos_cn: double
n_zeros_cn: double
media_cn: double
media_positivas_cn: double
mediana_cn: double
std_cn: double
q1_cn: do

,tabela,arquivo_MB,linhas,colunas,row_groups
0,painel,1.185326,6464,82,1
1,faixa,0.144585,63836,7,1
2,grupo,8.573952,337214,16,1
3,dsp,10.102821,451205,17,1


## 4. Ponto 1 — custo da leitura integral

Esta etapa lê uma tabela por vez, mede o tempo, o tamanho efetivo do DataFrame e a variação do RSS do processo. A leitura integral pode exigir memória adicional temporária durante a descompressão e conversão.

In [4]:
resultados_leitura = []
dados_integral = {}

for nome, caminho in ARQUIVOS.items():
    if not caminho.exists():
        continue

    gc.collect()
    rss_antes = memoria_processo_mb()
    inicio = time.perf_counter()

    df = pd.read_parquet(caminho)

    tempo = time.perf_counter() - inicio
    rss_depois = memoria_processo_mb()
    memoria_df = tamanho_df_mb(df)

    dados_integral[nome] = df
    resultados_leitura.append({
        "tabela": nome,
        "linhas": len(df),
        "colunas": len(df.columns),
        "DataFrame_MB": memoria_df,
        "RSS_antes_MB": rss_antes,
        "RSS_depois_MB": rss_depois,
        "delta_RSS_MB": rss_depois - rss_antes if pd.notna(rss_antes) and pd.notna(rss_depois) else np.nan,
        "tempo_s": tempo,
    })

    print(f"{nome}: {len(df):,} linhas | {len(df.columns)} colunas | "
          f"{memoria_df:,.1f} MB | {tempo:.2f} s")
    del df

resultado_leitura_df = pd.DataFrame(resultados_leitura)
display(resultado_leitura_df)
registrar_memoria("Após leitura integral das tabelas")

painel: 6,464 linhas | 82 colunas | 4.3 MB | 0.03 s
faixa: 63,836 linhas | 7 colunas | 5.2 MB | 0.01 s
grupo: 337,214 linhas | 16 colunas | 55.5 MB | 0.04 s
dsp: 451,205 linhas | 17 colunas | 76.0 MB | 0.04 s


,tabela,linhas,colunas,DataFrame_MB,RSS_antes_MB,RSS_depois_MB,delta_RSS_MB,tempo_s
0,painel,6464,82,4.280939,148.226562,191.507812,43.281250,0.033075
1,faixa,63836,7,5.183252,191.820312,228.492188,36.671875,0.006764
2,grupo,337214,16,55.466128,228.492188,373.953125,145.460938,0.037409
3,dsp,451205,17,76.008500,373.953125,476.406250,102.453125,0.042061


Após leitura integral das tabelas             RSS=476.4 MB


476.40625

## 5. Ponto 1 — identificar colunas úteis e testar projeção

Confira abaixo os nomes e tipos. Depois, preencha `COLUNAS_TESTE` com as colunas necessárias para uma página ou operação específica. Não escolha campos por suposição: confirme o uso nos módulos de `pages/` e em `utils/features_names.py`.

O teste compara a leitura integral com a leitura apenas das colunas selecionadas.

In [5]:
for nome, schema in esquemas.items():
    print(f"\n{nome}")
    display(pd.DataFrame({
        "coluna": schema.names,
        "tipo_arrow": [str(schema.field(c).type) for c in schema.names],
    }))


painel


,coluna,tipo_arrow
0,ano,int64
1,CO_MUNICIPIO_TERRITORIO,large_string
2,NO_MUNICIPIO_TERRITORIO,large_string
3,TIPO_REFERENCIA_TERRITORIAL,large_string
4,n_participantes,int64
...,...,...
77,pct_escola_federal,double
78,pct_escola_estadual,double
79,pct_escola_municipal,double
80,pct_escola_privada,double



faixa


,coluna,tipo_arrow
0,ano,int64
1,CO_MUNICIPIO_TERRITORIO,large_string
2,NO_MUNICIPIO_TERRITORIO,large_string
3,TIPO_REFERENCIA_TERRITORIAL,large_string
4,TP_FAIXA_ETARIA,large_string
5,n_participantes,int64
6,base_origem,large_string



grupo


,coluna,tipo_arrow
0,ano,int64
1,CO_MUNICIPIO_TERRITORIO,large_string
2,NO_MUNICIPIO_TERRITORIO,large_string
3,TIPO_REFERENCIA_TERRITORIAL,large_string
4,tipo_grupo,large_string
5,grupo,large_string
6,disciplina,large_string
7,n_validos,int64
8,n_zeros,int64
9,media,double



dsp


,coluna,tipo_arrow
0,ano,int64
1,CO_MUNICIPIO_TERRITORIO,large_string
2,NO_MUNICIPIO_TERRITORIO,large_string
3,TIPO_REFERENCIA_TERRITORIAL,large_string
4,sexo,large_string
5,faixa_etaria,large_string
6,disciplina,large_string
7,n_participantes,int64
8,n_validos,int64
9,n_zeros,int64


In [6]:
# PREENCHER após inspecionar as colunas e os usos no código.
COLUNAS_TESTE = {
    "painel": [
        "ano", "CO_MUNICIPIO_TERRITORIO",
        "TIPO_REFERENCIA_TERRITORIAL",
        "n_participantes", "n_masculino", "n_feminino",
        "n_faixa_etaria_informada",
        "n_validos_redacao", "media_redacao",
        "n_validos_objetiva", "media_objetiva",
        "n_validos_cn", "media_cn",
        "n_validos_ch", "media_ch",
        "n_validos_lc", "media_lc",
        "n_validos_mt", "media_mt",
        "n_escola_federal", "n_escola_estadual",
        "n_escola_municipal", "n_escola_privada",
        "n_escola_nao_informada",
    ],
    "faixa": [
        "ano", "CO_MUNICIPIO_TERRITORIO",
        "TIPO_REFERENCIA_TERRITORIAL",
        "TP_FAIXA_ETARIA", "n_participantes",
    ],
    "grupo": [
        "ano", "CO_MUNICIPIO_TERRITORIO",
        "TIPO_REFERENCIA_TERRITORIAL",
        "tipo_grupo", "grupo", "disciplina",
        "n_validos", "n_zeros", "media",
        "media_positivas", "mediana", "std", "q1", "q3",
    ],
    "dsp": [
        "ano", "CO_MUNICIPIO_TERRITORIO",
        "TIPO_REFERENCIA_TERRITORIAL",
        "sexo", "faixa_etaria", "disciplina",
        "n_participantes", "n_validos", "n_zeros",
        "media", "media_positivas", "mediana", "std", "q1", "q3",
    ],
}

# Validação para evitar nomes inexistentes.
for tabela, colunas in COLUNAS_TESTE.items():
    if tabela not in esquemas:
        raise KeyError(f"Tabela não encontrada: {tabela}")
    inexistentes = sorted(set(colunas) - set(esquemas[tabela].names))
    if inexistentes:
        raise ValueError(f"{tabela}: colunas inexistentes: {inexistentes}")

print("Configuração de projeção validada.")

Configuração de projeção validada.


In [7]:
print("COLUNAS_TESTE:")
for tabela, colunas in COLUNAS_TESTE.items():
    print(tabela, len(colunas), colunas)

print("\nTabelas disponíveis:")
print(ARQUIVOS.keys())

print("\nTabelas carregadas integralmente:")
print(dados_integral.keys())

COLUNAS_TESTE:
painel 24 ['ano', 'CO_MUNICIPIO_TERRITORIO', 'TIPO_REFERENCIA_TERRITORIAL', 'n_participantes', 'n_masculino', 'n_feminino', 'n_faixa_etaria_informada', 'n_validos_redacao', 'media_redacao', 'n_validos_objetiva', 'media_objetiva', 'n_validos_cn', 'media_cn', 'n_validos_ch', 'media_ch', 'n_validos_lc', 'media_lc', 'n_validos_mt', 'media_mt', 'n_escola_federal', 'n_escola_estadual', 'n_escola_municipal', 'n_escola_privada', 'n_escola_nao_informada']
faixa 5 ['ano', 'CO_MUNICIPIO_TERRITORIO', 'TIPO_REFERENCIA_TERRITORIAL', 'TP_FAIXA_ETARIA', 'n_participantes']
grupo 14 ['ano', 'CO_MUNICIPIO_TERRITORIO', 'TIPO_REFERENCIA_TERRITORIAL', 'tipo_grupo', 'grupo', 'disciplina', 'n_validos', 'n_zeros', 'media', 'media_positivas', 'mediana', 'std', 'q1', 'q3']
dsp 15 ['ano', 'CO_MUNICIPIO_TERRITORIO', 'TIPO_REFERENCIA_TERRITORIAL', 'sexo', 'faixa_etaria', 'disciplina', 'n_participantes', 'n_validos', 'n_zeros', 'media', 'media_positivas', 'mediana', 'std', 'q1', 'q3']

Tabelas disponí

In [8]:
resultados_projecao = []

for nome, colunas in COLUNAS_TESTE.items():
    if not colunas:
        continue

    caminho = ARQUIVOS[nome]
    gc.collect()
    rss_antes = memoria_processo_mb()
    inicio = time.perf_counter()

    df_reduzido = pd.read_parquet(caminho, columns=colunas)

    tempo = time.perf_counter() - inicio
    rss_depois = memoria_processo_mb()
    mb_reduzido = tamanho_df_mb(df_reduzido)
    mb_integral = tamanho_df_mb(dados_integral[nome])

    resultados_projecao.append({
        "tabela": nome,
        "colunas_integral": len(dados_integral[nome].columns),
        "colunas_selecionadas": len(colunas),
        "integral_MB": mb_integral,
        "projetado_MB": mb_reduzido,
        "reducao_MB": mb_integral - mb_reduzido,
        "reducao_pct": (1 - mb_reduzido / mb_integral) * 100 if mb_integral else np.nan,
        "tempo_s": tempo,
        "delta_RSS_MB": rss_depois - rss_antes if pd.notna(rss_antes) and pd.notna(rss_depois) else np.nan,
    })

    del df_reduzido

resultado_projecao_df = pd.DataFrame(resultados_projecao)
display(resultado_projecao_df)

,tabela,colunas_integral,colunas_selecionadas,integral_MB,projetado_MB,reducao_MB,reducao_pct,tempo_s,delta_RSS_MB
0,painel,82,24,4.280939,1.298841,2.982099,69.659916,0.013526,6.304688
1,faixa,7,5,5.183252,3.244058,1.939195,37.412701,0.004313,6.500000
2,grupo,16,14,55.466128,45.319761,10.146367,18.292907,0.024872,74.652344
3,dsp,17,15,76.008500,62.505886,13.502614,17.764611,0.031013,23.472656


## 6. Ponto 1 — avaliar filtros no Parquet antes de carregar

O código abaixo lista os campos disponíveis para verificar se existe uma coluna de referência (`ESC`/`PROVA`) ou outro campo que permita reduzir os registros ainda na leitura.

Apenas identifica candidatos. O filtro só será testado depois de confirmar o nome do campo e os valores válidos.

In [9]:
for nome, schema in esquemas.items():
    candidatos = [
        c for c in schema.names
        if any(chave in c.lower() for chave in ("refer", "prova", "esc", "tipo", "depend"))
    ]
    print(f"{nome}: {candidatos}")
for tabela, df in dados_integral.items():
    print(f"\n{tabela}:")
    print(sorted(df["TIPO_REFERENCIA_TERRITORIAL"].dropna().unique()))

painel: ['TIPO_REFERENCIA_TERRITORIAL', 'n_registros_escola', 'n_escola_federal', 'n_escola_estadual', 'n_escola_municipal', 'n_escola_privada', 'n_escola_nao_informada', 'n_dependencia_informada', 'pct_escola_federal', 'pct_escola_estadual', 'pct_escola_municipal', 'pct_escola_privada', 'pct_escola_nao_informada']
faixa: ['TIPO_REFERENCIA_TERRITORIAL']
grupo: ['TIPO_REFERENCIA_TERRITORIAL', 'tipo_grupo']
dsp: ['TIPO_REFERENCIA_TERRITORIAL']

painel:
['ESC', 'PROVA', 'RESIDENCIA']

faixa:
['ESC', 'PROVA', 'RESIDENCIA']

grupo:
['ESC', 'PROVA', 'RESIDENCIA']

dsp:
['ESC', 'PROVA']


In [10]:
# Preencher somente após confirmar coluna e valores na etapa anterior.
# Exemplo estrutural:
FILTRO_TESTE = {
    "painel": ("TIPO_REFERENCIA_TERRITORIAL", "ESC"),
    "faixa": ("TIPO_REFERENCIA_TERRITORIAL", "ESC"),
    "grupo": ("TIPO_REFERENCIA_TERRITORIAL", "ESC"),
    "dsp": ("TIPO_REFERENCIA_TERRITORIAL", "ESC"),
}


# FILTRO_TESTE = {
#     "painel": ("TIPO_REFERENCIA_TERRITORIAL", "PROVA"),
#     "faixa": ("TIPO_REFERENCIA_TERRITORIAL", "PROVA"),
#     "grupo": ("TIPO_REFERENCIA_TERRITORIAL", "PROVA"),
#     "dsp": ("TIPO_REFERENCIA_TERRITORIAL", "PROVA"),
# }

# Leitura filtrada via PyArrow Dataset; não altera os arquivos.
import pyarrow.dataset as ds

resultados_filtro = []
for nome, (coluna, valor) in FILTRO_TESTE.items():
    if nome not in ARQUIVOS or not ARQUIVOS[nome].exists():
        raise KeyError(f"Tabela inexistente: {nome}")
    if coluna not in esquemas[nome].names:
        raise KeyError(f"Coluna {coluna} não existe em {nome}")

    dataset = ds.dataset(ARQUIVOS[nome], format="parquet")
    filtro = ds.field(coluna) == valor

    inicio = time.perf_counter()
    tabela_filtrada = dataset.to_table(filter=filtro)
    df_filtrado = tabela_filtrada.to_pandas()
    tempo = time.perf_counter() - inicio

    resultados_filtro.append({
        "tabela": nome,
        "filtro": f"{coluna} == {valor}",
        "linhas_filtradas": len(df_filtrado),
        "colunas": len(df_filtrado.columns),
        "DataFrame_MB": tamanho_df_mb(df_filtrado),
        "tempo_s": tempo,
    })
    del tabela_filtrada, df_filtrado, dataset
    gc.collect()

display(pd.DataFrame(resultados_filtro))

,tabela,filtro,linhas_filtradas,colunas,DataFrame_MB,tempo_s
0,painel,TIPO_REFERENCIA_TERRITORIAL == ESC,4982,82,3.297628,0.007796
1,faixa,TIPO_REFERENCIA_TERRITORIAL == ESC,39879,7,3.195717,0.005235
2,grupo,TIPO_REFERENCIA_TERRITORIAL == ESC,203374,16,33.341311,0.029559
3,dsp,TIPO_REFERENCIA_TERRITORIAL == ESC,263761,17,44.310968,0.041182


## 7. Ponto 2 — custo das cópias dos DataFrames

O carregador do projeto já utiliza cache e entrega cópias profundas (`copy(deep=True)`) das tabelas para proteger os objetos armazenados. Esta etapa quantifica o custo dessa estratégia por tabela e no conjunto.

A medição é feita com os dados integrais já carregados na seção 4.

In [11]:
resultados_copias = []
copias_teste = {}

for nome, df in dados_integral.items():
    gc.collect()
    rss_antes = memoria_processo_mb()
    inicio = time.perf_counter()

    copia = df.copy(deep=True)

    tempo = time.perf_counter() - inicio
    rss_depois = memoria_processo_mb()

    resultados_copias.append({
        "tabela": nome,
        "original_MB": tamanho_df_mb(df),
        "copia_MB": tamanho_df_mb(copia),
        "tempo_copia_s": tempo,
        "delta_RSS_MB": rss_depois - rss_antes if pd.notna(rss_antes) and pd.notna(rss_depois) else np.nan,
    })
    copias_teste[nome] = copia

display(pd.DataFrame(resultados_copias))
registrar_memoria("Com originais + cópias profundas")

,tabela,original_MB,copia_MB,tempo_copia_s,delta_RSS_MB
0,painel,4.280939,4.280939,0.000944,1.148438
1,faixa,5.183252,5.183252,0.002751,1.082031
2,grupo,55.466128,55.466128,0.006059,23.476562
3,dsp,76.008500,76.008500,0.007618,34.859375


Com originais + cópias profundas              RSS=717.5 MB


717.45703125

## 8. Ponto 2 — custo de manter todas as tabelas simultaneamente

Compara o tamanho efetivo das quatro tabelas e estima a memória ocupada por uma cópia adicional de cada uma. A soma de `memory_usage` é uma estimativa do conteúdo dos DataFrames; o RSS do processo inclui bibliotecas, caches e outras alocações.

In [12]:
resumo_objetos = []
for nome, df in dados_integral.items():
    resumo_objetos.append({
        "tabela": nome,
        "linhas": len(df),
        "colunas": len(df.columns),
        "memoria_MB": tamanho_df_mb(df),
    })

resumo_objetos_df = pd.DataFrame(resumo_objetos)
display(resumo_objetos_df)

total_df_mb = resumo_objetos_df["memoria_MB"].sum()
print(f"Total estimado das tabelas: {total_df_mb:,.1f} MB")
print(f"Total estimado com uma cópia adicional: {2 * total_df_mb:,.1f} MB")
registrar_memoria("Após manter tabelas e cópias")

,tabela,linhas,colunas,memoria_MB
0,painel,6464,82,4.280939
1,faixa,63836,7,5.183252
2,grupo,337214,16,55.466128
3,dsp,451205,17,76.008500


Total estimado das tabelas: 140.9 MB
Total estimado com uma cópia adicional: 281.9 MB
Após manter tabelas e cópias                  RSS=717.5 MB


717.48046875

## 9. Ponto 2 — localizar operações que criam cópias/intermediários

Busca textual nos módulos do dashboard por padrões comuns de cópia e transformação. A busca é indicativa: cada ocorrência precisa ser interpretada no contexto do código.

In [13]:
DIRETORIOS_CODIGO = [RAIZ / "dash_utils", RAIZ / "pages", RAIZ / "src"]
PADROES = (".copy(", "copy(deep=True)", "merge(", "concat(", "reset_index(", "to_dict(")

ocorrencias = []
for diretorio in DIRETORIOS_CODIGO:
    if not diretorio.exists():
        continue
    for arquivo in diretorio.rglob("*.py"):
        try:
            linhas = arquivo.read_text(encoding="utf-8").splitlines()
        except UnicodeDecodeError:
            continue
        for numero, linha in enumerate(linhas, start=1):
            if any(padrao in linha for padrao in PADROES):
                ocorrencias.append({
                    "arquivo": str(arquivo.relative_to(RAIZ)),
                    "linha": numero,
                    "codigo": linha.strip(),
                })

ocorrencias_df = pd.DataFrame(ocorrencias)
pd.set_option('display.max_rows', None)

display(ocorrencias_df)

,arquivo,linha,codigo
0,dash_utils/data_loader.py,114,valor.copy(deep=False)
1,dash_utils/data_loader.bkp.py,78,"df = df[df[""TIPO_REFERENCIA_TERRITORIAL""] != o..."
2,dash_utils/data_loader.bkp.py,87,# chave: (valor.copy(deep=True) if cha...
3,dash_utils/data_loader.bkp.py,94,chave: (valor.copy(deep=False) if chave in _TA...
4,pages/perfil_participantes.py,101,df = painel.copy()
5,pages/perfil_participantes.py,122,"return df.dropna(subset=[""ano""]).copy()"
6,pages/perfil_participantes.py,126,df = faixa.copy()
7,pages/perfil_participantes.py,147,"df = df.dropna(subset=[""ano"", ""faixa"", ""n_part..."
8,pages/perfil_participantes.py,155,df = grupo.copy()
9,pages/perfil_participantes.py,181,").loc[lambda x: x[""n_validos""] > 0].copy()"


## 10. Limpeza e encerramento

In [14]:
# Remover objetos auxiliares de teste para liberar a memória do kernel.
copias_teste.clear()
dados_integral.clear()
gc.collect()

registrar_memoria("Após liberar objetos do notebook")

Após liberar objetos do notebook              RSS=694.7 MB


694.67578125

## 11. Síntese dos resultados

Preencha após executar as etapas. Não conclua que uma técnica é melhor apenas pelo tamanho do arquivo: compare memória efetiva, tempo e impacto no comportamento do dashboard.

### Carregamento
- Tabelas com maior consumo:
- Ganho observado com seleção de colunas:
- Filtros que podem ser aplicados durante a leitura:

### Processamento
- Custo das cópias profundas:
- Operações que criam objetos intermediários:
- Principais oportunidades de redução:

### Próxima etapa
Com os resultados, decidir quais mudanças serão feitas no `data_loader.py` e nos módulos de páginas. Este notebook não altera o código de produção.

In [15]:
import sys
from pathlib import Path

# Disponibilizar os módulos do projeto
SRC_DIR = Path.cwd() / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Importar a função atualizada
from utils.features_names import read_allfiles2

# Testar as três modalidades
for referencia in [None, "ESC", "PROVA"]:

    print(f"\n{'=' * 50}")
    print(f"REFERÊNCIA: {referencia}")

    dados = read_allfiles2("PA", P=referencia)

    for tabela in ["painel", "faixa", "grupo", "dsp"]:
        df = dados[tabela]["data"]

        print(
            f"{tabela:8s} | "
            f"Linhas: {len(df):8,d} | "
            f"Referências: {df['TIPO_REFERENCIA_TERRITORIAL'].unique().tolist()}"
        )

    del dados


REFERÊNCIA: None
painel   | Linhas:    4,964 | Referências: ['ESC', 'PROVA']
faixa    | Linhas:   61,512 | Referências: ['ESC', 'PROVA']
grupo    | Linhas:  318,263 | Referências: ['ESC', 'PROVA']
dsp      | Linhas:  437,880 | Referências: ['ESC', 'PROVA']

REFERÊNCIA: ESC
painel   | Linhas:    3,625 | Referências: ['ESC']
faixa    | Linhas:   38,335 | Referências: ['ESC']
grupo    | Linhas:  186,449 | Referências: ['ESC']
dsp      | Linhas:  250,436 | Referências: ['ESC']

REFERÊNCIA: PROVA
painel   | Linhas:    1,339 | Referências: ['PROVA']
faixa    | Linhas:   23,177 | Referências: ['PROVA']
grupo    | Linhas:  131,814 | Referências: ['PROVA']
dsp      | Linhas:  187,444 | Referências: ['PROVA']


In [16]:
import gc
import time
import psutil
import pandas as pd

processo = psutil.Process()

gc.collect()
rss_antes = processo.memory_info().rss / 1024**2

inicio = time.perf_counter()

dados = read_allfiles2("PA", P=None)

tempo = time.perf_counter() - inicio
rss_depois = processo.memory_info().rss / 1024**2

memoria_tabelas = sum(
    dados[nome]["data"].memory_usage(deep=True).sum()
    for nome in ["painel", "faixa", "grupo", "dsp"]
) / 1024**2

print(f"Tempo de leitura: {tempo:.3f} s")
print(f"Memória dos DataFrames: {memoria_tabelas:.2f} MB")
print(f"RSS antes: {rss_antes:.2f} MB")
print(f"RSS depois: {rss_depois:.2f} MB")
print(f"Variação RSS: {rss_depois - rss_antes:.2f} MB")

Tempo de leitura: 0.259 s
Memória dos DataFrames: 101.40 MB
RSS antes: 730.86 MB
RSS depois: 821.92 MB
Variação RSS: 91.06 MB


In [17]:
import sys
import gc
import time
import psutil
from pathlib import Path

DASH_DIR = Path.cwd() / "dash"

if str(DASH_DIR) not in sys.path:
    sys.path.insert(0, str(DASH_DIR))

from dash_utils.data_loader import (
    carregar_dados,
    limpar_cache_dados,
    status_cache_dados
)

processo = psutil.Process()

# Limpar cache anterior
limpar_cache_dados()
gc.collect()

rss_antes = processo.memory_info().rss / 1024**2

# Primeira chamada: leitura do disco
inicio = time.perf_counter()
dados = carregar_dados("PA", referencia=None)
tempo_primeira = time.perf_counter() - inicio

rss_depois = processo.memory_info().rss / 1024**2

print("PRIMEIRA CHAMADA")
print(f"Tempo: {tempo_primeira:.3f} s")
print(f"RSS antes: {rss_antes:.2f} MB")
print(f"RSS depois: {rss_depois:.2f} MB")

print("\nLINHAS CARREGADAS")
for tabela in ["painel", "faixa", "grupo", "dsp"]:
    print(f"{tabela}: {len(dados[tabela]):,}")

# Segunda chamada: deve aproveitar o cache
inicio = time.perf_counter()
dados2 = carregar_dados("PA", referencia=None)
tempo_segunda = time.perf_counter() - inicio

print(f"\nSEGUNDA CHAMADA: {tempo_segunda:.4f} s")

print("\nSTATUS DO CACHE")
print(status_cache_dados())

PRIMEIRA CHAMADA
Tempo: 0.263 s
RSS antes: 821.98 MB
RSS depois: 907.53 MB

LINHAS CARREGADAS
painel: 4,964
faixa: 61,512
grupo: 318,263
dsp: 437,880

SEGUNDA CHAMADA: 0.0001 s

STATUS DO CACHE
{'entradas_em_memoria': [{'uf': 'PA', 'referencia': 'ESC+PROVA'}], 'quantidade': 1, 'limite': 3}


In [18]:
import pandas as pd
import gc
import psutil

processo = psutil.Process()

print("VERSÃO DO PANDAS:", pd.__version__)
print("COPY-ON-WRITE:", pd.options.mode.copy_on_write)

print("\nMEMÓRIA DOS OBJETOS")

for nome in ["painel", "faixa", "grupo", "dsp"]:
    df = dados[nome]

    memoria = df.memory_usage(deep=True).sum() / 1024**2

    print(
        f"{nome:8s} | "
        f"Linhas: {len(df):>9,} | "
        f"Colunas: {len(df.columns):>3} | "
        f"Memória: {memoria:8.2f} MB"
    )

shape = dados["shape"]

print(
    f"{'shape':8s} | "
    f"Linhas: {len(shape):>9,} | "
    f"Colunas: {len(shape.columns):>3} | "
    f"Memória: {shape.memory_usage(deep=True).sum() / 1024**2:8.2f} MB"
)

gc.collect()

print(
    f"\nRSS atual: "
    f"{processo.memory_info().rss / 1024**2:.2f} MB"
)

VERSÃO DO PANDAS: 3.1.0
COPY-ON-WRITE: True

MEMÓRIA DOS OBJETOS
painel   | Linhas:     4,964 | Colunas:  24 | Memória:     1.05 MB


/tmp/ipykernel_110714/3556568139.py:8: Pandas4Warning: The 'mode.copy_on_write' option is deprecated. Copy-on-Write can no longer be disabled (it is always enabled with pandas >= 3.0), and setting the option has no impact. This option will be removed in pandas 4.0.
  print("COPY-ON-WRITE:", pd.options.mode.copy_on_write)


faixa    | Linhas:    61,512 | Colunas:   6 | Memória:     4.26 MB
grupo    | Linhas:   318,263 | Colunas:  13 | Memória:    43.91 MB
dsp      | Linhas:   437,880 | Colunas:  11 | Memória:    52.18 MB
shape    | Linhas:       144 | Colunas:  16 | Memória:     0.03 MB

RSS atual: 907.54 MB


In [19]:
for referencia in ["ESC", "PROVA"]:

    gc.collect()

    rss_antes = processo.memory_info().rss / 1024**2
    inicio = time.perf_counter()

    dados_teste = read_allfiles2("PA", P=referencia)

    tempo = time.perf_counter() - inicio
    rss_depois = processo.memory_info().rss / 1024**2

    memoria = sum(
        dados_teste[nome]["data"].memory_usage(deep=True).sum()
        for nome in ["painel", "faixa", "grupo", "dsp"]
    ) / 1024**2

    print(f"\nREFERÊNCIA: {referencia}")
    print(f"Tempo: {tempo:.3f} s")
    print(f"Memória dos DataFrames: {memoria:.2f} MB")
    print(f"Variação RSS: {rss_depois - rss_antes:.2f} MB")

    for nome in ["painel", "faixa", "grupo", "dsp"]:
        df = dados_teste[nome]["data"]
        print(f"{nome}: {len(df):,} linhas")

    del dados_teste


REFERÊNCIA: ESC
Tempo: 0.192 s
Memória dos DataFrames: 58.82 MB
Variação RSS: 26.20 MB
painel: 3,625 linhas
faixa: 38,335 linhas
grupo: 186,449 linhas
dsp: 250,436 linhas

REFERÊNCIA: PROVA
Tempo: 0.112 s
Memória dos DataFrames: 42.58 MB
Variação RSS: 21.47 MB
painel: 1,339 linhas
faixa: 23,177 linhas
grupo: 131,814 linhas
dsp: 187,444 linhas
